In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain.tools import tool, ToolRuntime

@tool
def read_email(runtime: ToolRuntime) -> str:
    """read an email from the given address."""
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """send an email to the given address with the given subject and body."""
    return f"Email sent."

In [3]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

class EmailState(AgentState):
    email: str

agent = create_agent(
    model="claude-haiku-4-5",
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True
            },
            description_prefix="Tool execution requires approval"
        )
    ]
)

In [4]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Read my email and send a response to the sender. Draft a professional response on your own.")],
        "email": "Hellos, I am unable to make the meeting tomorrow. Can we reschedule? Regards, Ash."
    },
    config = config
)

In [5]:
from pprint import pprint
pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Dear '
                                                                          'Ash,\n'
                                                                          '\n'
                                                                          'Thank '
                                                                          'you '
                                                                          'for '
                                                                          'letting '
                                                                          'me '
                                                                          'know '
                                                                          'that '
                                                                          'you '
                                                                          'are '
                    

In [6]:
print(response['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': "Dear Ash,\n\nThank you for letting me know that you are unable to attend tomorrow's meeting. I appreciate you reaching out with advance notice.\n\nI would be happy to reschedule at a time that works better for you. Please let me know your availability, and I will do my best to accommodate your schedule.\n\nLooking forward to connecting soon.\n\nBest regards"}, 'description': 'Tool execution requires approval\n\nTool: send_email\nArgs: {\'body\': "Dear Ash,\\n\\nThank you for letting me know that you are unable to attend tomorrow\'s meeting. I appreciate you reaching out with advance notice.\\n\\nI would be happy to reschedule at a time that works better for you. Please let me know your availability, and I will do my best to accommodate your schedule.\\n\\nLooking forward to connecting soon.\\n\\nBest regards"}'}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reje

In [7]:
# access the 'body' argument from the tool call
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Dear Ash,

Thank you for letting me know that you are unable to attend tomorrow's meeting. I appreciate you reaching out with advance notice.

I would be happy to reschedule at a time that works better for you. Please let me know your availability, and I will do my best to accommodate your schedule.

Looking forward to connecting soon.

Best regards


In [8]:
# in case of editing the response. 
from langgraph.types import Command

response = agent.invoke(
    Command(
        resume={
            "decisions": [{
                "type": "edit", 
                "edited_action": {
                    "name": "send_email",
                    "args": {"body": "Your unprofessional behaviour is not tolerated. You are fired."}
                }
            }]
        }
    ),
    config = config
)

pprint(response)

{'email': 'Hellos, I am unable to make the meeting tomorrow. Can we '
          'reschedule? Regards, Ash.',
 'messages': [HumanMessage(content='Read my email and send a response to the sender. Draft a professional response on your own.', additional_kwargs={}, response_metadata={}, id='511c696a-3050-4503-8fd0-f7ba70c562b7'),
              AIMessage(content=[{'text': "I'll read your email first to see what needs to be responded to.", 'type': 'text'}, {'id': 'toolu_01VFQkuP8ZV1Nc23nQjYzi18', 'input': {}, 'name': 'read_email', 'type': 'tool_use', 'caller': {'type': 'direct'}}], additional_kwargs={}, response_metadata={'id': 'msg_01LviHDc6RrrfMtdUFdq12HQ', 'model': 'claude-haiku-4-5-20251001', 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'input_tokens': 624, 'output_tokens': 52, 'server_tool_use': None, 'service_tier': 'standard'